# 🔌 Smart Transformer Health Monitoring System
### RPPL Transformers — Machine Learning Pipeline
---
**Dataset:** 5 sensor CSV files | ~19,000 readings each | June 2019 – April 2020

**Goal:** Predict transformer health status → **Normal / Warning / Critical**

**Pipeline Overview:**
1. Load & Explore Data
2. Merge All Files
3. Exploratory Data Analysis (EDA)
4. Feature Engineering
5. Health Labeling
6. Model Training (Random Forest + XGBoost)
7. Model Evaluation
8. Anomaly Detection
9. Feature Importance
10. Save Model & Predict on New Data


## 📦 Step 1 — Install & Import Libraries

In [ ]:
# Run this once if needed
# !pip install pandas numpy scikit-learn xgboost matplotlib seaborn joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from functools import reduce
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, roc_auc_score)
from sklearn.preprocessing import StandardScaler, label_binarize
import xgboost as xgb
import joblib

# Plot style
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print("✅ All libraries loaded successfully!")

## 📂 Step 2 — Load the Dataset

> **Put all 5 CSV files in the same folder as this notebook before running.**

| File | Contents |
|------|----------|
| `CurrentVoltage.csv` | Per-phase voltage (V) and current (A) |
| `Overview.csv` | Oil/winding temperature, oil level, alarms |
| `Power.csv` | Active, apparent & reactive power per phase |
| `PowerFactor.csv` | Power factor, frequency, THD harmonics |
| `TotalPower.csv` | Cumulative energy, KW, KVA, KVAR |


In [ ]:
# ── Load all 5 files ──────────────────────────────────────────
cv = pd.read_csv("CurrentVoltage.csv")
ov = pd.read_csv("Overview.csv")
pw = pd.read_csv("Power.csv")
pf = pd.read_csv("PowerFactor.csv")
tp = pd.read_csv("TotalPower.csv")

files = {'CurrentVoltage': cv, 'Overview': ov,
         'Power': pw, 'PowerFactor': pf, 'TotalPower': tp}

# Parse timestamps
for df in files.values():
    df['DeviceTimeStamp'] = pd.to_datetime(df['DeviceTimeStamp'])

# Summary
print(f"{'File':<20} {'Rows':>8} {'Cols':>6}  {'Date Range'}")
print("-" * 70)
for name, df in files.items():
    ts = df['DeviceTimeStamp']
    print(f"{name:<20} {df.shape[0]:>8,} {df.shape[1]:>6}  "
          f"{ts.min().date()} → {ts.max().date()}")

## 👀 Step 3 — Quick Peek at Each File

In [ ]:
for name, df in files.items():
    print(f"\n{'='*60}")
    print(f"  {name}")
    print('='*60)
    print(f"  Columns : {df.columns.tolist()}")
    print(f"  Nulls   : {df.isnull().sum().sum()}")
    display(df.head(3))

## 🔗 Step 4 — Merge All Files

All files share `DeviceTimeStamp`. We merge them into one master DataFrame using an **outer join** so no reading is lost.


In [ ]:
data = reduce(lambda a, b: pd.merge(a, b, on='DeviceTimeStamp', how='outer'),
              [cv, ov, pw, pf, tp])

data = data.sort_values('DeviceTimeStamp').reset_index(drop=True)
data = data.ffill().fillna(0)   # forward-fill gaps, then zero

print(f"✅ Merged shape  : {data.shape[0]:,} rows × {data.shape[1]} columns")
print(f"   Date range    : {data['DeviceTimeStamp'].min()} → {data['DeviceTimeStamp'].max()}")
print(f"   Remaining nulls: {data.isnull().sum().sum()}")
data.head(3)

## 📊 Step 5 — Exploratory Data Analysis (EDA)

We'll visualize key sensor signals to understand normal behaviour and spot anomalies.


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
fig.suptitle("Transformer Sensor Signals Over Time", fontsize=15, fontweight='bold')

plots = [
    ('VL1',    'Phase L1 Voltage (V)',          'royalblue'),
    ('IL1',    'Phase L1 Current (A)',           'darkorange'),
    ('OTI',    'Oil Temperature Index (°C)',     'crimson'),
    ('OLI',    'Oil Level Index',                'seagreen'),
    ('Avg_PF', 'Average Power Factor',           'purple'),
    ('KW',     'Active Power (kW)',              'saddlebrown'),
]

for ax, (col, title, color) in zip(axes.flat, plots):
    if col in data.columns:
        ax.plot(data['DeviceTimeStamp'], data[col], color=color, linewidth=0.6, alpha=0.8)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel('')
        ax.tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.savefig("eda_signals.png", bbox_inches='tight')
plt.show()
print("Saved: eda_signals.png")

In [ ]:
# ── Statistical summary of key columns ───────────────────────
key_cols = ['VL1','VL2','VL3','IL1','IL2','IL3',
            'OTI','OLI','Avg_PF','KW','KVA','KVAR']
key_cols = [c for c in key_cols if c in data.columns]

print("📋 Statistical Summary:")
display(data[key_cols].describe().round(2))

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────
plt.figure(figsize=(14, 10))
corr = data[key_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={'size': 8})
plt.title("Feature Correlation Heatmap", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("correlation_heatmap.png", bbox_inches='tight')
plt.show()
print("Saved: correlation_heatmap.png")

## ⚙️ Step 6 — Feature Engineering

We create domain-specific features that directly capture transformer stress and health signals.

| Feature | Formula | Why it matters |
|---|---|---|
| `V_imbalance` | std(VL1,VL2,VL3) / mean × 100 | >2% causes extra heating |
| `I_imbalance` | std(IL1,IL2,IL3) / mean × 100 | >10% indicates unequal loading |
| `Thermal_stress` | OTI / OTI_max | Normalised oil heat index |
| `PF_drop` | 1 - |Avg_PF| | Low PF = poor efficiency |
| `Load_level` | KW / KW_max | Normalised loading |
| `OTI_trend` | rolling(OTI, 10) | Rising temp trend |
| `I_total` | IL1+IL2+IL3 | Total current draw |
| `Power_loss_ratio` | KVAR / (KVA+0.001) | Reactive power waste |


In [ ]:
# ── Voltage & Current imbalance ───────────────────────────────
v_mean = data[['VL1','VL2','VL3']].mean(axis=1)
v_std  = data[['VL1','VL2','VL3']].std(axis=1)
data['V_imbalance'] = (v_std / (v_mean + 0.001)) * 100

i_mean = data[['IL1','IL2','IL3']].mean(axis=1)
i_std  = data[['IL1','IL2','IL3']].std(axis=1)
data['I_imbalance'] = (i_std / (i_mean + 0.001)) * 100

# ── Thermal stress ────────────────────────────────────────────
data['Thermal_stress'] = data['OTI'] / (data['OTI'].max() + 0.001)

# ── Power factor drop ─────────────────────────────────────────
data['PF_drop'] = 1 - data['Avg_PF'].abs()

# ── Load level ────────────────────────────────────────────────
data['Load_level'] = data['KW'] / (data['KW'].max() + 0.001)

# ── Rolling temperature trend (last 10 readings ~10 min) ──────
data['OTI_trend']  = data['OTI'].rolling(10, min_periods=1).mean()
data['OTI_change'] = data['OTI'].diff().fillna(0)  # rate of temp change

# ── Total current ─────────────────────────────────────────────
data['I_total'] = data['IL1'] + data['IL2'] + data['IL3']

# ── Power loss ratio ──────────────────────────────────────────
data['Power_loss_ratio'] = data['KVAR'] / (data['KVA'] + 0.001)

# ── Hour of day (operational pattern) ────────────────────────
data['Hour'] = data['DeviceTimeStamp'].dt.hour
data['DayOfWeek'] = data['DeviceTimeStamp'].dt.dayofweek

print("✅ Engineered features added:")
new_feats = ['V_imbalance','I_imbalance','Thermal_stress','PF_drop',
             'Load_level','OTI_trend','OTI_change','I_total',
             'Power_loss_ratio','Hour','DayOfWeek']
display(data[new_feats].describe().round(3))

## 🏷️ Step 7 — Health Status Labeling

Since we have no pre-labeled faults, we apply **rule-based labeling** based on industry standards (IEC 60076, IEEE C57):

| Status | Condition |
|---|---|
| 🟢 Normal (0) | All parameters within safe operating range |
| 🟡 Warning (1) | One or more parameters approaching limits |
| 🔴 Critical (2) | Overtemperature, high imbalance, or low PF |


In [ ]:
def get_health_label(row):
    """
    Returns:
        0 = Normal
        1 = Warning
        2 = Critical
    """
    # ── Critical ────────────────────────────────────────────
    if (row['OTI'] > 80                 # oil overtemperature
        or row['V_imbalance'] > 5       # severe voltage imbalance
        or row['I_imbalance'] > 20      # severe current imbalance
        or row['Avg_PF'] < 0.80         # very low power factor
        or row['IL1'] > 200             # overcurrent phase 1
        or row['IL2'] > 200             # overcurrent phase 2
        or row['IL3'] > 200):           # overcurrent phase 3
        return 2

    # ── Warning ─────────────────────────────────────────────
    elif (row['OTI'] > 55
          or row['V_imbalance'] > 2
          or row['I_imbalance'] > 10
          or row['Avg_PF'] < 0.90
          or row['IL1'] > 150
          or row['IL2'] > 150
          or row['IL3'] > 150):
        return 1

    # ── Normal ──────────────────────────────────────────────
    else:
        return 0

data['health'] = data.apply(get_health_label, axis=1)

# ── Distribution ──────────────────────────────────────────────
label_map   = {0: 'Normal', 1: 'Warning', 2: 'Critical'}
label_colors = {0: '#2ecc71', 1: '#f39c12', 2: '#e74c3c'}
counts = data['health'].value_counts().sort_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
bars = ax1.bar([label_map[k] for k in counts.index],
               counts.values,
               color=[label_colors[k] for k in counts.index],
               edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'{val:,}\n({val/len(data)*100:.1f}%)',
             ha='center', fontsize=10)
ax1.set_title("Health Status Distribution", fontweight='bold')
ax1.set_ylabel("Number of Readings")

# Pie chart
ax2.pie(counts.values,
        labels=[label_map[k] for k in counts.index],
        colors=[label_colors[k] for k in counts.index],
        autopct='%1.1f%%', startangle=140,
        wedgeprops=dict(edgecolor='white', linewidth=2))
ax2.set_title("Health Status Proportion", fontweight='bold')

plt.tight_layout()
plt.savefig("health_distribution.png", bbox_inches='tight')
plt.show()

print("\n📊 Label counts:")
for k, v in counts.items():
    print(f"   {label_map[k]:<10}: {v:>6,} readings ({v/len(data)*100:.1f}%)")

## 🎯 Step 8 — Prepare Features & Train/Test Split

In [ ]:
FEATURES = [
    # Raw sensor readings
    'VL1', 'VL2', 'VL3',
    'IL1', 'IL2', 'IL3',
    'OTI', 'OLI',
    'Avg_PF', 'FRQ',
    'THDVL1', 'THDVL2', 'THDVL3',
    'THDIL1', 'THDIL2', 'THDIL3',
    'KW', 'KVA', 'KVAR',
    'KWH',
    # Engineered features
    'V_imbalance', 'I_imbalance',
    'Thermal_stress', 'PF_drop',
    'Load_level', 'OTI_trend', 'OTI_change',
    'I_total', 'Power_loss_ratio',
    'Hour', 'DayOfWeek'
]

# Keep only columns that exist in data
FEATURES = [f for f in FEATURES if f in data.columns]
TARGET   = 'health'

X = data[FEATURES]
y = data[TARGET]

print(f"✅ Features used  : {len(FEATURES)}")
print(f"   Total samples  : {len(X):,}")
print(f"   Features list  : {FEATURES}")

# ── Time-based split (80% train, 20% test) ────────────────────
# We split by time — NOT randomly — because that's how real deployment works.
split_idx = int(len(data) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"\n✂️  Train : {len(X_train):,} samples  "
      f"({data['DeviceTimeStamp'].iloc[0].date()} → "
      f"{data['DeviceTimeStamp'].iloc[split_idx].date()})")
print(f"   Test  : {len(X_test):,} samples  "
      f"({data['DeviceTimeStamp'].iloc[split_idx].date()} → "
      f"{data['DeviceTimeStamp'].iloc[-1].date()})")

## 🌲 Step 9 — Random Forest Classifier

Random Forest is an ensemble of decision trees. It's robust, fast, and gives feature importances — perfect for this kind of multi-sensor tabular data.


In [ ]:
print("🌲 Training Random Forest...")

rf = RandomForestClassifier(
    n_estimators=150,     # number of trees
    max_depth=12,         # max depth per tree
    min_samples_split=5,
    random_state=42,
    n_jobs=-1             # use all CPU cores
)
rf.fit(X_train, y_train)

# ── Predictions ───────────────────────────────────────────────
rf_pred = rf.predict(X_test)
rf_acc  = accuracy_score(y_test, rf_pred)

print(f"\n✅ Random Forest Accuracy : {rf_acc*100:.2f}%")
print("\n📋 Classification Report:")
print(classification_report(y_test, rf_pred,
      target_names=['Normal','Warning','Critical']))

## ⚡ Step 10 — XGBoost Classifier

XGBoost (Extreme Gradient Boosting) often outperforms Random Forest on tabular data. It builds trees sequentially, correcting errors at each step.


In [ ]:
print("⚡ Training XGBoost...")

xgb_model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)],
              verbose=False)

xgb_pred = xgb_model.predict(X_test)
xgb_acc  = accuracy_score(y_test, xgb_pred)

print(f"\n✅ XGBoost Accuracy : {xgb_acc*100:.2f}%")
print("\n📋 Classification Report:")
print(classification_report(y_test, xgb_pred,
      target_names=['Normal','Warning','Critical']))

## 📊 Step 11 — Model Comparison & Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels = ['Normal', 'Warning', 'Critical']
cmaps  = ['Blues', 'Greens']
models = [('Random Forest', rf_pred), ('XGBoost', xgb_pred)]

for ax, cmap, (name, pred) in zip(axes, cmaps, models):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=labels, yticklabels=labels,
                linewidths=0.5, ax=ax)
    acc = accuracy_score(y_test, pred)
    ax.set_title(f"{name}\nAccuracy: {acc*100:.2f}%", fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.suptitle("Confusion Matrices — Model Comparison", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("confusion_matrices.png", bbox_inches='tight')
plt.show()

# Accuracy comparison bar
fig, ax = plt.subplots(figsize=(6, 4))
model_names = ['Random Forest', 'XGBoost']
accs = [accuracy_score(y_test, rf_pred)*100, accuracy_score(y_test, xgb_pred)*100]
bars = ax.bar(model_names, accs, color=['royalblue', 'darkorange'],
              edgecolor='white', linewidth=1.5, width=0.5)
for bar, val in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 2,
            f'{val:.2f}%', ha='center', color='white', fontweight='bold', fontsize=12)
ax.set_ylim(90, 101)
ax.set_title("Model Accuracy Comparison", fontweight='bold')
ax.set_ylabel("Accuracy (%)")
plt.tight_layout()
plt.savefig("model_comparison.png", bbox_inches='tight')
plt.show()

## 🔁 Step 12 — Cross Validation (Model Reliability Check)

In [ ]:
print("Running 5-fold cross-validation (this may take ~1 minute)...")

rf_cv  = cross_val_score(rf,        X, y, cv=5, scoring='accuracy', n_jobs=-1)
xgb_cv = cross_val_score(xgb_model, X, y, cv=5, scoring='accuracy', n_jobs=-1)

print(f"\n🌲 Random Forest CV:")
print(f"   Scores : {[f'{s*100:.2f}%' for s in rf_cv]}")
print(f"   Mean   : {rf_cv.mean()*100:.2f}%  ±  {rf_cv.std()*100:.2f}%")

print(f"\n⚡ XGBoost CV:")
print(f"   Scores : {[f'{s*100:.2f}%' for s in xgb_cv]}")
print(f"   Mean   : {xgb_cv.mean()*100:.2f}%  ±  {xgb_cv.std()*100:.2f}%")

## 🔍 Step 13 — Feature Importance

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Random Forest
rf_imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True).tail(15)
rf_imp.plot(kind='barh', ax=ax1, color='royalblue', edgecolor='white')
ax1.set_title("Random Forest — Top 15 Features", fontweight='bold')
ax1.set_xlabel("Importance Score")

# XGBoost
xgb_imp = pd.Series(xgb_model.feature_importances_, index=FEATURES).sort_values(ascending=True).tail(15)
xgb_imp.plot(kind='barh', ax=ax2, color='darkorange', edgecolor='white')
ax2.set_title("XGBoost — Top 15 Features", fontweight='bold')
ax2.set_xlabel("Importance Score")

plt.suptitle("Feature Importance Comparison", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("feature_importance.png", bbox_inches='tight')
plt.show()
print("Saved: feature_importance.png")

## 🚨 Step 14 — Anomaly Detection (Unsupervised)

**Isolation Forest** finds unusual readings without needing labels. It isolates anomalies by randomly splitting data — anomalies are isolated faster because they're few and different.


In [ ]:
print("🔍 Training Isolation Forest for anomaly detection...")

iso = IsolationForest(
    contamination=0.05,   # expect ~5% anomalies
    n_estimators=100,
    random_state=42
)
iso.fit(X_train)

anomaly_scores = iso.decision_function(X_test)
anomaly_preds  = iso.predict(X_test)   # -1 = anomaly, 1 = normal

n_anomalies = (anomaly_preds == -1).sum()
print(f"\n⚠️  Anomalies detected : {n_anomalies:,} / {len(X_test):,} "
      f"({n_anomalies/len(X_test)*100:.1f}%)")

# ── Visualize anomaly scores over time ───────────────────────
test_timestamps = data['DeviceTimeStamp'].iloc[split_idx:].reset_index(drop=True)

plt.figure(figsize=(15, 4))
plt.plot(test_timestamps, anomaly_scores, color='steelblue', linewidth=0.6, alpha=0.7, label='Anomaly Score')
plt.axhline(0, color='red', linewidth=1, linestyle='--', label='Anomaly Threshold')
anomaly_mask = anomaly_preds == -1
plt.scatter(test_timestamps[anomaly_mask], anomaly_scores[anomaly_mask],
            color='red', s=8, alpha=0.5, label=f'Anomaly ({n_anomalies:,})')
plt.title("Isolation Forest — Anomaly Detection Over Time", fontweight='bold')
plt.xlabel("Timestamp")
plt.ylabel("Anomaly Score (lower = more anomalous)")
plt.legend()
plt.tight_layout()
plt.savefig("anomaly_detection.png", bbox_inches='tight')
plt.show()
print("Saved: anomaly_detection.png")

## 📅 Step 15 — Transformer Health Timeline

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)
fig.suptitle("Transformer Health Monitoring Dashboard", fontsize=14, fontweight='bold')

color_map = {0: '#2ecc71', 1: '#f39c12', 2: '#e74c3c'}
health_colors = data['health'].map(color_map)

# Panel 1: Health status scatter
ax = axes[0]
for status, color, name in [(0,'#2ecc71','Normal'),(1,'#f39c12','Warning'),(2,'#e74c3c','Critical')]:
    mask = data['health'] == status
    ax.scatter(data.loc[mask,'DeviceTimeStamp'], [status]*mask.sum(),
               c=color, s=4, alpha=0.6, label=name)
ax.set_yticks([0,1,2])
ax.set_yticklabels(['Normal','Warning','Critical'])
ax.set_title("Health Status", fontweight='bold')
ax.legend(loc='upper right', markerscale=3)

# Panel 2: Oil temperature
ax = axes[1]
ax.plot(data['DeviceTimeStamp'], data['OTI'], color='crimson', linewidth=0.6)
ax.axhline(55, color='orange', linestyle='--', linewidth=1, label='Warning (55°C)')
ax.axhline(80, color='red',    linestyle='--', linewidth=1, label='Critical (80°C)')
ax.set_title("Oil Temperature Index (OTI)", fontweight='bold')
ax.set_ylabel("°C")
ax.legend(loc='upper right')

# Panel 3: Voltage imbalance
ax = axes[2]
ax.fill_between(data['DeviceTimeStamp'], data['V_imbalance'],
                color='royalblue', alpha=0.5)
ax.axhline(2, color='orange', linestyle='--', linewidth=1, label='Warning (2%)')
ax.axhline(5, color='red',    linestyle='--', linewidth=1, label='Critical (5%)')
ax.set_title("Voltage Imbalance (%)", fontweight='bold')
ax.set_ylabel("%")
ax.legend(loc='upper right')

# Panel 4: Active power
ax = axes[3]
ax.plot(data['DeviceTimeStamp'], data['KW'], color='saddlebrown', linewidth=0.6)
ax.set_title("Active Power (kW)", fontweight='bold')
ax.set_ylabel("kW")
ax.set_xlabel("Date")

plt.tight_layout()
plt.savefig("health_timeline.png", bbox_inches='tight')
plt.show()
print("Saved: health_timeline.png")

## 💾 Step 16 — Save Model & Predict on New Data

In [ ]:
# ── Save best model ───────────────────────────────────────────
best_model = rf if accuracy_score(y_test, rf_pred) >= accuracy_score(y_test, xgb_pred) else xgb_model
best_name  = "RandomForest" if best_model is rf else "XGBoost"

joblib.dump(best_model, "transformer_health_model.pkl")
joblib.dump(FEATURES,   "feature_names.pkl")

print(f"✅ Best model ({best_name}) saved as: transformer_health_model.pkl")
print(f"✅ Feature list saved as           : feature_names.pkl")

In [ ]:
# ── Predict on a new reading ──────────────────────────────────
# Replace these values with real-time sensor readings

new_reading = {
    'VL1': 242.0, 'VL2': 241.5, 'VL3': 243.0,
    'IL1': 95.0,  'IL2': 97.0,  'IL3': 94.0,
    'OTI': 45.0,  'OLI': 37.0,
    'Avg_PF': 0.98, 'FRQ': 50.0,
    'THDVL1': 1.2,  'THDVL2': 1.1, 'THDVL3': 1.3,
    'THDIL1': 3.5,  'THDIL2': 3.2, 'THDIL3': 3.8,
    'KW': 55.0, 'KVA': 57.0, 'KVAR': 12.0,
    'KWH': 1200.0,
    'V_imbalance': 0.3,  'I_imbalance': 1.5,
    'Thermal_stress': 0.18, 'PF_drop': 0.02,
    'Load_level': 0.39, 'OTI_trend': 44.0, 'OTI_change': 0.5,
    'I_total': 286.0, 'Power_loss_ratio': 0.21,
    'Hour': 14, 'DayOfWeek': 2
}

# Load saved model
loaded_model   = joblib.load("transformer_health_model.pkl")
loaded_features = joblib.load("feature_names.pkl")

new_df = pd.DataFrame([new_reading])
new_df = new_df[[f for f in loaded_features if f in new_df.columns]]

prediction = loaded_model.predict(new_df)[0]
proba      = loaded_model.predict_proba(new_df)[0]
status     = {0: '🟢 NORMAL', 1: '🟡 WARNING', 2: '🔴 CRITICAL'}

print("=" * 45)
print("  TRANSFORMER HEALTH PREDICTION")
print("=" * 45)
print(f"  Status     : {status[prediction]}")
print(f"  Normal     : {proba[0]*100:.1f}%")
print(f"  Warning    : {proba[1]*100:.1f}%")
print(f"  Critical   : {proba[2]*100:.1f}%")
print("=" * 45)

if prediction == 2:
    print("\n  ⚡ ACTION: Schedule immediate inspection!")
elif prediction == 1:
    print("\n  ⚠️  ACTION: Monitor closely, plan maintenance.")
else:
    print("\n  ✅ ACTION: No action needed. Continue monitoring.")

## ✅ Step 17 — Project Summary

### What we built:
1. **Loaded & merged** 5 transformer sensor CSVs (~31,000 readings)
2. **Engineered features**: voltage/current imbalance, thermal stress, power factor drop, rolling trends
3. **Labeled** health status: Normal / Warning / Critical using IEC/IEEE-based rules
4. **Trained two models**: Random Forest & XGBoost
5. **Detected anomalies** using Isolation Forest (unsupervised)
6. **Visualized** sensor trends, health timeline, confusion matrices, feature importance
7. **Saved** the best model for real-time prediction

### Output files:
| File | Description |
|---|---|
| `transformer_health_model.pkl` | Trained ML model |
| `feature_names.pkl` | Feature list for prediction |
| `eda_signals.png` | Sensor signals over time |
| `correlation_heatmap.png` | Feature correlations |
| `health_distribution.png` | Label distribution |
| `confusion_matrices.png` | Model evaluation |
| `feature_importance.png` | Top predictive features |
| `anomaly_detection.png` | Isolation Forest results |
| `health_timeline.png` | Full health monitoring dashboard |

### Next steps:
- 🔄 **Real-time pipeline**: Connect to live SCADA/IoT sensor feed
- 📱 **Alert system**: Send SMS/email when status = Warning or Critical
- 🌐 **Web dashboard**: Deploy with Flask/Streamlit for plant operators
- 📈 **Remaining Useful Life (RUL)**: Add regression model to predict time-to-failure
